# Customer Segmentation & Personalized Recommendation System

**Dataset:** UCI Online Retail Dataset  
**Dataset URL:** https://archive.ics.uci.edu/ml/datasets/Online+Retail  
**Download:** https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx

---

## Project Overview

This notebook walks through an end-to-end data science project:

1. Data Loading & Cleaning
2. Exploratory Data Analysis (EDA)
3. Feature Engineering (RFM + Behavioral)
4. Dimensionality Reduction (PCA)
5. Customer Segmentation (K-Means + Hierarchical)
6. Recommendation System (Segment-based + Similarity-based)
7. Business Insights & Dashboard


## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Point to the project root
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline
print('Libraries loaded ✓')

## 1. Data Loading

**Before running this cell**, download the dataset from:
```
https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx
```
and save it to `../data/online_retail.xlsx`.

Alternatively, run `python main.py` from the project root — it attempts an automatic download.

In [ ]:
from src.data_loader import load_raw_data, get_column_info

df_raw = load_raw_data()
get_column_info(df_raw)
df_raw.head()

In [ ]:
print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw["InvoiceDate"].min()} → {df_raw["InvoiceDate"].max()}')
print(f'Countries: {df_raw["Country"].nunique()}')
df_raw.describe()

## 2. Data Cleaning & Feature Engineering

In [ ]:
from src.preprocessing import (
    clean_raw_data, compute_rfm,
    compute_behavioral_features, merge_customer_profile
)

df = clean_raw_data(df_raw)
rfm = compute_rfm(df)
behavioral = compute_behavioral_features(df)
profile = merge_customer_profile(rfm, behavioral)

print('\nRFM Table (first 5 rows):')
rfm.head()

In [ ]:
print('Customer Profile (first 5 rows):')
profile.head()

## 3. Exploratory Data Analysis

In [ ]:
from src.eda import (
    plot_sales_over_time, plot_top_products, plot_geographic_distribution,
    plot_hourly_orders, plot_dow_orders, plot_rfm_distributions,
    plot_rfm_correlation, plot_spending_boxplots
)

plot_sales_over_time(df)

In [ ]:
plot_top_products(df, top_n=12)

In [ ]:
plot_geographic_distribution(df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Hourly
hourly = df.groupby('Hour')['InvoiceNo'].nunique().reset_index()
axes[0].bar(hourly['Hour'], hourly['InvoiceNo'], color='steelblue', alpha=0.8)
axes[0].set_title('Orders by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Orders')
# Day of week
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('DayOfWeek')['InvoiceNo'].nunique().reindex(order)
axes[1].bar(dow.index, dow.values, color='coral', alpha=0.8)
axes[1].set_title('Orders by Day of Week', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plot_rfm_distributions(rfm)
plot_rfm_correlation(rfm)

## 4. Feature Preparation & PCA

In [ ]:
from src.feature_engineering import prepare_features
from src.pca_analysis import run_pca_pipeline

X_scaled, feature_names, scaler = prepare_features(profile)
print(f'Features: {feature_names}')
print(f'Scaled shape: {X_scaled.shape}')

In [ ]:
X_pca, pca_model = run_pca_pipeline(X_scaled, feature_names, variance_threshold=0.85)
print(f'PCA output shape: {X_pca.shape}')

## 5. Customer Segmentation

In [ ]:
from src.clustering import (
    compute_cluster_metrics, plot_cluster_selection,
    fit_kmeans, analyse_clusters, run_clustering_pipeline
)

# Evaluate cluster counts
metrics = compute_cluster_metrics(X_pca, range(2, 10))
best_k = plot_cluster_selection(metrics)
print(f'\nOptimal k: {best_k}')

In [ ]:
# Run full pipeline (includes Hierarchical, radar, distribution plots)
profile, best_k, segment_map = run_clustering_pipeline(
    X_scaled, X_pca, profile, k_range=range(2, 10)
)

print('\nSegment distribution:')
print(profile['Segment'].value_counts())

In [ ]:
# Interactive PCA scatter
fig = px.scatter(
    x=X_pca[:, 0], y=X_pca[:, 1],
    color=profile['Segment'],
    title='Customer Segments in PCA Space',
    labels={'x': 'PC1', 'y': 'PC2'},
    opacity=0.6,
    width=900, height=600
)
fig.show()

In [ ]:
# Segment stats
cols = [c for c in ['Recency','Frequency','Monetary','AvgOrderValue','UniqueProducts'] if c in profile.columns]
seg_stats = profile.groupby('Segment')[cols].mean().round(1)
seg_stats['Count'] = profile.groupby('Segment').size()
seg_stats['Revenue%'] = (profile.groupby('Segment')['Monetary'].sum() / profile['Monetary'].sum() * 100).round(1)
seg_stats.sort_values('Monetary', ascending=False)

## 6. Recommendation System

In [ ]:
from src.recommendation import run_recommendation_pipeline

rec_results = run_recommendation_pipeline(df, profile, top_n=10)

print('\nRecommendation coverage:')
print(rec_results['coverage'].to_string(index=False))

In [ ]:
# Segment-based recommendations example
segment_recs = rec_results['segment_recs']
for seg, recs in list(segment_recs.items())[:2]:
    print(f'\n=== {seg} ===')
    print(recs[['Description','Revenue','PurchaseCount']].head(5).to_string(index=False))

In [ ]:
# Similarity-based recommendations for a sample customer
recommender = rec_results['recommender']
sample_cid = profile['CustomerID'].iloc[0]
seg = profile.loc[profile['CustomerID'] == sample_cid, 'Segment'].iloc[0]
recs = recommender.recommend(sample_cid, top_n=8)

print(f'Customer {sample_cid} — Segment: {seg}')
print(recs[['StockCode','Description','Score']].to_string(index=False))

In [ ]:
# Popularity baseline
print('Global Popularity Baseline:')
print(rec_results['popularity'][['Description','UniqueBuyers','Revenue']].to_string(index=False))

## 7. Interactive Dashboard

In [ ]:
from src.visualization import build_executive_dashboard, generate_business_report

dashboard_path = build_executive_dashboard(
    df=df, profile=profile, X_pca=X_pca,
    segment_recs=rec_results['segment_recs'],
    popularity=rec_results['popularity'],
    coverage=rec_results['coverage'],
)

report_path = generate_business_report(
    profile=profile,
    segment_recs=rec_results['segment_recs'],
    coverage=rec_results['coverage'],
    best_k=best_k,
)

print(f'Dashboard: {dashboard_path}')
print(f'Report:    {report_path}')

In [ ]:
# Render dashboard inline
from IPython.display import IFrame
IFrame(src='../outputs/reports/dashboard.html', width='100%', height=900)

---

## Summary

| Step | Output |
|------|--------|
| Data Cleaning | Removed cancelled orders, anonymous customers, noise |
| RFM Features | Recency, Frequency, Monetary per customer |
| Behavioral Features | Avg order value, product diversity, purchase span |
| PCA | Reduced to ≥85% variance in fewer dimensions |
| K-Means | Optimal k via Elbow + Silhouette + DB Index |
| Hierarchical | Ward linkage, dendrogram visualisation |
| Segment-Based Recs | Top products per business segment |
| Similarity-Based Recs | Cosine similarity recommender |
| Dashboard | Interactive HTML at `outputs/reports/dashboard.html` |
| Business Report | Markdown at `outputs/reports/business_report.md` |
